# Calendar-aware New Year indicators — demo

Compares the legacy `t % 52` placement of `ny_pre` / `ny_mid` / `ny_post` against the calendar-aware version in `pyjags_pipeline.data_calendar_proto`.

**Why a fix is needed.** `t % 52` assumes a year is exactly 364 days. The Gregorian year is ~365.25 days, so the indicator drifts off Jan 1 within one year. The calendar-aware version anchors `ny_mid` on the W-SUN week that actually contains Jan 1 of each year covered by the data.

**Convention.** Every column label is the Sunday end-date of a Mon..Sun week.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

for _candidate in (Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent):
    if (_candidate / 'pyjags_pipeline').exists():
        sys.path.insert(0, str(_candidate))
        break

from pyjags_pipeline.data_calendar_proto import build_event_indicators

### Helpers

- `w_sun_range(start, n)` builds `n` consecutive W-SUN Sunday end-dates as strings.
- `compare(dates)` runs both legacy and calendar-aware indicators and returns a side-by-side `DataFrame`.

In [2]:
def w_sun_range(start, n):
    return pd.date_range(start=start, periods=n, freq='W-SUN').strftime('%Y-%m-%d').tolist()


def compare(dates, regions=('HSE Mid West',)):
    n = len(dates)
    leg = build_event_indicators(n, list(regions), column_dates=None)
    cal = build_event_indicators(n, list(regions), column_dates=dates)
    rows = []
    for tag in ('ny_pre', 'ny_mid', 'ny_post'):
        leg_d = [dates[i] for i, v in enumerate(leg[tag]) if v == 1]
        cal_d = [dates[i] for i, v in enumerate(cal[tag]) if v == 1]
        rows.append({'indicator': tag, 'legacy': leg_d, 'calendar': cal_d})
    return pd.DataFrame(rows).set_index('indicator')

## Case 1 — canonical project layout (~3 years, 156 weeks)

NY 2023 falls on Sunday, 2024 on Monday, 2025 on Wednesday.

**What to look for in the table below:**
- legacy `ny_mid` lands on `2023-01-01` (correct only by coincidence — 2023 NY was Sunday, so it sits in the W-SUN week labelled by Jan 1 itself), then on `2023-12-31` and `2024-12-29` — both **off by one week**: those are pre-NY weeks, not the NY weeks themselves.
- calendar `ny_mid` lands on the actual NY-containing weeks: `2024-01-07` and `2025-01-05`.
- legacy `ny_pre` has 3 entries (`2023-12-24`, `2024-12-22`, `2025-12-21`), but only 2 real NYs have a `pre` week inside the data range — the third is a phantom flag at the last column (`t=156, t%52==0`) that doesn't correspond to any actual New Year.

In [3]:
dates = w_sun_range('2023-01-01', 156)
compare(dates)

,legacy,calendar
indicator,,
ny_pre,"[2023-12-24, 2024-12-22, 2025-12-21]","[2023-12-31, 2024-12-29]"
ny_mid,"[2023-01-01, 2023-12-31, 2024-12-29]","[2023-01-01, 2024-01-07, 2025-01-05]"
ny_post,"[2023-01-08, 2024-01-07, 2025-01-05]","[2023-01-08, 2024-01-14, 2025-01-12]"


## Case 2 — 5-year span (261 weeks)

The drift compounds. Watch the **counts**: legacy returns 5/6/5 indicators (pre/mid/post) but calendar returns 4/5/5 — there are only 5 real Jan-1 events in the range, so legacy is over-flagging by one in both `ny_pre` and `ny_mid`. Each year after 2023 also lands on the wrong calendar week.

In [4]:
dates = w_sun_range('2023-01-01', 261)
compare(dates)

,legacy,calendar
indicator,,
ny_pre,"[2023-12-24, 2024-12-22, 2025-12-21, 2026-12-2...","[2023-12-31, 2024-12-29, 2025-12-28, 2026-12-27]"
ny_mid,"[2023-01-01, 2023-12-31, 2024-12-29, 2025-12-2...","[2023-01-01, 2024-01-07, 2025-01-05, 2026-01-0..."
ny_post,"[2023-01-08, 2024-01-07, 2025-01-05, 2026-01-0...","[2023-01-08, 2024-01-14, 2025-01-12, 2026-01-1..."


## Case 3 — mid-year start (110 weeks)

Synthetic stress test: data starts Sun 2024-06-02, runs ~2 years.

**What to look for:** legacy is anchored on `t`, not the calendar, so the very first column (`t=1`, June 2024) is flagged as `ny_mid` and the indicators are placed on **summer dates** that have no relationship to New Year. Calendar correctly places `ny_mid` on `2025-01-05` and `2026-01-04` with proper `pre`/`post` flanks.

This case isn't realistic for the production pipeline (which always starts in January), but it makes the bug unmistakable: the legacy indicators are a function of array index, not of dates.

In [5]:
dates = w_sun_range('2024-06-02', 110)
compare(dates)

,legacy,calendar
indicator,,
ny_pre,"[2025-05-25, 2026-05-24]","[2024-12-29, 2025-12-28]"
ny_mid,"[2024-06-02, 2025-06-01, 2026-05-31]","[2025-01-05, 2026-01-04]"
ny_post,"[2024-06-09, 2025-06-08, 2026-06-07]","[2025-01-12, 2026-01-11]"


## Case 4 — data ends ON a NY week (53 weeks)

Data starts Sun 2024-01-07 (a real NY 2024 week) and ends Sun 2025-01-05 (a real NY 2025 week), exactly 52 weeks apart.

**What to look for:** legacy and calendar agree **exactly** here. This is not because the legacy is correct — it's a coincidence. The data starts on a real NY week, so `t=1` happens to be the right column for `ny_mid`, and 52 weeks later (`t=53`) is also a real NY week. NY 2024 was a Monday (so the W-SUN week starting Mon Jan 1 ends Sun Jan 7), and NY 2025 falls in the W-SUN week ending Sun Jan 5 — the exact 52-week stride between them is what makes legacy look right.

The case still validates the **right-boundary guard** in the calendar logic: `ny_mid` is set for `t=53`, but `ny_post` at `t=54` would overshoot the data (`n_weeks=53`), so it is correctly omitted.

In [6]:
dates = w_sun_range('2024-01-07', 53)
compare(dates)

,legacy,calendar
indicator,,
ny_pre,[2024-12-29],[2024-12-29]
ny_mid,"[2024-01-07, 2025-01-05]","[2024-01-07, 2025-01-05]"
ny_post,[2024-01-14],[2024-01-14]


## Drift summary

For the 5-year span: every row legacy emits, paired with the matching calendar row by index (year), so phantom legacy flags with no calendar counterpart are visible. `drift_weeks > 0` means legacy is too **early**; `(phantom)` means legacy flagged a column that has no corresponding Jan 1 in the data.

In [7]:
from itertools import zip_longest

dates = w_sun_range('2023-01-01', 261)
n = len(dates)
leg = build_event_indicators(n, ['HSE Mid West'], column_dates=None)
cal = build_event_indicators(n, ['HSE Mid West'], column_dates=dates)
leg_mid = [dates[i] for i in np.where(leg['ny_mid'] == 1)[0]]
cal_mid = [dates[i] for i in np.where(cal['ny_mid'] == 1)[0]]

drift_rows = []
for ld, cd in zip_longest(leg_mid, cal_mid):
    if ld is None:
        drift_rows.append({'legacy_mid': '(missing)', 'calendar_mid': cd, 'drift_weeks': '-'})
    elif cd is None:
        drift_rows.append({'legacy_mid': ld, 'calendar_mid': '(phantom)', 'drift_weeks': '-'})
    else:
        drift = (pd.Timestamp(cd) - pd.Timestamp(ld)).days // 7
        drift_rows.append({'legacy_mid': ld, 'calendar_mid': cd, 'drift_weeks': drift})

pd.DataFrame(drift_rows)

,legacy_mid,calendar_mid,drift_weeks
0,2023-01-01,2023-01-01,0
1,2023-12-31,2024-01-07,1
2,2024-12-29,2025-01-05,1
3,2025-12-28,2026-01-04,1
4,2026-12-27,2027-01-03,1
5,2027-12-26,(phantom),-


## Validation checks

Hard pass/fail assertions for each case. If any of these fail, the calendar-aware logic regressed.

In [8]:
results = []

def check(name, condition):
    results.append((name, bool(condition)))
    print(('PASS' if condition else 'FAIL') + f'  {name}')


# Case 1
dates = w_sun_range('2023-01-01', 156)
cal = build_event_indicators(len(dates), ['HSE Mid West'], column_dates=dates)
mid = {dates[i] for i in np.where(cal['ny_mid'] == 1)[0]}
check('Case 1: ny_mid on Jan-1 weeks for 2023/24/25',
      mid == {'2023-01-01', '2024-01-07', '2025-01-05'})

# Case 3
dates = w_sun_range('2024-06-02', 110)
cal = build_event_indicators(len(dates), ['HSE Mid West'], column_dates=dates)
mid = [dates[i] for i in np.where(cal['ny_mid'] == 1)[0]]
check('Case 3: finds 2 NY events (2025-01-05, 2026-01-04)',
      mid == ['2025-01-05', '2026-01-04'])
check('Case 3: 2 pre flanks and 2 post flanks',
      int(cal['ny_pre'].sum()) == 2 and int(cal['ny_post'].sum()) == 2)

# Case 4
dates = w_sun_range('2024-01-07', 53)
cal = build_event_indicators(len(dates), ['HSE Mid West'], column_dates=dates)
post_dates = [dates[i] for i in np.where(cal['ny_post'] == 1)[0]]
check('Case 4: ny_post does not overshoot end of data',
      '2025-01-12' not in post_dates)

all_ok = all(ok for _, ok in results)
print()
print('All PASS' if all_ok else 'SOME FAILED')

PASS  Case 1: ny_mid on Jan-1 weeks for 2023/24/25
PASS  Case 3: finds 2 NY events (2025-01-05, 2026-01-04)
PASS  Case 3: 2 pre flanks and 2 post flanks
PASS  Case 4: ny_post does not overshoot end of data

All PASS


## Per-week ground-truth test

Stronger validation: build a DataFrame with **one row per week** for the 5-year span. For each row, independently compute the expected indicator values from the calendar — no call to `build_event_indicators` — then compare row-for-row against what the function returns.

The independent rule:
- `expected_mid` is `True` iff the row's Mon..Sun window contains Jan 1 of *some* year.
- `expected_pre` is `expected_mid.shift(-1)` (the row before a `mid` is a `pre`).
- `expected_post` is `expected_mid.shift(1)`.

If `cal_*` matches `expected_*` on every row, the function is correct on every week, not just the ones we picked out.

In [ ]:
def contains_jan1(week_start, week_end):
    """True iff the Mon..Sun window [week_start, week_end] contains Jan 1 of any year."""
    for yr in range(week_start.year, week_end.year + 1):
        jan1 = pd.Timestamp(yr, 1, 1)
        if week_start <= jan1 <= week_end:
            return True
    return False


def per_week_table(start, n):
    """Build a per-week DataFrame: legacy + calendar + independently-derived expected."""
    dates = pd.to_datetime(w_sun_range(start, n))
    date_strings = [d.strftime('%Y-%m-%d') for d in dates]
    leg = build_event_indicators(n, ['HSE Mid West'], column_dates=None)
    cal = build_event_indicators(n, ['HSE Mid West'], column_dates=date_strings)

    df = pd.DataFrame({
        't':           np.arange(1, n + 1),
        'date':        date_strings,
        'week_start':  (dates - pd.Timedelta(days=6)).strftime('%Y-%m-%d'),
        'week_end':    [d.strftime('%Y-%m-%d') for d in dates],
        'leg_pre':     leg['ny_pre'].astype(bool),
        'leg_mid':     leg['ny_mid'].astype(bool),
        'leg_post':    leg['ny_post'].astype(bool),
        'cal_pre':     cal['ny_pre'].astype(bool),
        'cal_mid':     cal['ny_mid'].astype(bool),
        'cal_post':    cal['ny_post'].astype(bool),
    })

    # Independent ground truth — never calls build_event_indicators.
    df['expected_mid']  = [contains_jan1(dates[i] - pd.Timedelta(days=6), dates[i])
                           for i in range(n)]
    df['expected_pre']  = df['expected_mid'].shift(-1, fill_value=False)
    df['expected_post'] = df['expected_mid'].shift( 1, fill_value=False)
    return df


df = per_week_table('2023-01-01', 261)

mismatches = df[(df['cal_pre']  != df['expected_pre']) |
                (df['cal_mid']  != df['expected_mid']) |
                (df['cal_post'] != df['expected_post'])]

print(f'rows total       : {len(df)}')
print(f'cal_mid set count: {int(df["cal_mid"].sum())}')
print(f'expected_mid sum : {int(df["expected_mid"].sum())}')
print(f'mismatches       : {len(mismatches)}  (PASS if 0)')
df.head()

### Inspect just the active rows

The full table has 261 rows; the interesting ones are the rows where any indicator is set. Filter to those.

In [ ]:
active = df[df[['leg_pre', 'leg_mid', 'leg_post',
                'cal_pre', 'cal_mid', 'cal_post',
                'expected_pre', 'expected_mid', 'expected_post']].any(axis=1)].copy()

# annotate each row with which indicators differ between legacy and calendar
def diff_tag(row):
    diffs = []
    for tag in ('pre', 'mid', 'post'):
        if row[f'leg_{tag}'] != row[f'cal_{tag}']:
            diffs.append(tag)
    return ','.join(diffs) if diffs else ''

active['legacy_vs_calendar_diff'] = active.apply(diff_tag, axis=1)
active[['t', 'date', 'leg_pre', 'leg_mid', 'leg_post',
        'cal_pre', 'cal_mid', 'cal_post',
        'expected_pre', 'expected_mid', 'expected_post',
        'legacy_vs_calendar_diff']]

### Repeat across multiple ranges

Run the same independent check across all four case ranges. Mismatch count must be 0 for every range.

In [ ]:
ranges = [
    ('Case 1: 2023-01-01 + 156w', '2023-01-01', 156),
    ('Case 2: 2023-01-01 + 261w', '2023-01-01', 261),
    ('Case 3: 2024-06-02 + 110w', '2024-06-02', 110),
    ('Case 4: 2024-01-07 +  53w', '2024-01-07',  53),
]

summary_rows = []
for name, start, n in ranges:
    df_case = per_week_table(start, n)
    bad = df_case[(df_case['cal_pre']  != df_case['expected_pre']) |
                  (df_case['cal_mid']  != df_case['expected_mid']) |
                  (df_case['cal_post'] != df_case['expected_post'])]
    summary_rows.append({
        'case': name,
        'rows': len(df_case),
        'cal_mid_count':       int(df_case['cal_mid'].sum()),
        'expected_mid_count':  int(df_case['expected_mid'].sum()),
        'mismatches':          len(bad),
        'pass':                len(bad) == 0,
    })

pd.DataFrame(summary_rows)